In [2]:
# https://github.com/h2oai/sql-sidekick/releases
# !python -m pip uninstall sql_sidekick-0.2.4-py3-none-any.whl -y

In [3]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [ ]:
from sidekick.prompter import db_setup, ask
from sidekick.schema_generator import generate_schema
from sidekick.utils import setup_dir, list_models

In [ ]:
base_path = "./"
cache_path = f"{base_path}/var/lib/tmp"
setup_dir(base_path)

In [ ]:
# import pandas as pd
# f = pd.read_csv("../demo/demo_data.csv")

In [ ]:
# Env variables
import os

os.environ['OPENAI_API_KEY'] = ""
os.environ['H2OGPT_URL'] = 'http://38.128.233.247'
os.environ['H2OGPT_API_TOKEN'] = ""
# To get access to h2ogpte endpoint, reach out to cloud-feedback@h2o.ai
os.environ['H2OGPTE_URL'] = "https://h2ogpte.genai.h2o.ai"  # e.g. https://<>.h2ogpte.h2o.ai
os.environ['H2OGPTE_API_TOKEN'] = ""

In [ ]:
HOST_NAME = "localhost"
USER_NAME = "sqlite"
PASSWORD = "abc"
DB_NAME = "querydb"
PORT = "5432"


# Given .csv file, auto-generate schema
# Download dataset --> https://www.kaggle.com/datasets/uom190346a/sleep-health-and-lifestyle-dataset
data_path = "examples/demo/demo_data.csv"
table_name = "sleep_health_eda"

r, table_info_path = generate_schema(data_path=data_path, output_path=f"{cache_path}/{table_name}_table_info.jsonl")

llm_response, err = db_setup(
                db_name=DB_NAME,
                hostname=HOST_NAME,
                user_name=USER_NAME,
                password=PASSWORD,
                port=PORT,
                table_info_path=table_info_path,
                table_samples_path=data_path,
                table_name=table_name,
                local_base_path=base_path
            )

 Information supplied:
 querydb, localhost, sqlite, abc, 5432
Database already exists!
Table name: sleep_health_eda


2024-06-12 16:44:00.837 | DEBUG    | sidekick.db_config:_extract_schema_info:162 - Using schema information from: .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:44:00.838 | DEBUG    | sidekick.db_config:create_table:186 - Schema info used for creating table:
 Person_ID NUMERIC,
Gender TEXT COLLATE NOCASE,
Age NUMERIC,
Occupation TEXT COLLATE NOCASE,
Sleep_Duration NUMERIC,
Quality_of_Sleep NUMERIC,
Physical_Activity_Level NUMERIC,
Stress_Level NUMERIC,
BMI_Category TEXT COLLATE NOCASE,
Blood_Pressure TEXT COLLATE NOCASE,
Heart_Rate NUMERIC,
Daily_Steps NUMERIC,
Sleep_Disorder TEXT COLLATE NOCASE
2024-06-12 16:44:00.839 | INFO     | sidekick.db_config:create_table:198 - Table created: sleep_health_eda


Checked table sleep_health_eda exists in the DB.


2024-06-12 16:44:00.841 | DEBUG    | sidekick.db_config:add_samples:222 - Adding sample values to table: examples/demo/demo_data.csv
2024-06-12 16:44:00.843 | DEBUG    | sidekick.db_config:add_samples:228 - Inserting chunk: 0
2024-06-12 16:44:01.024 | INFO     | sidekick.db_config:add_samples:233 - Data inserted into table: sleep_health_eda
2024-06-12 16:44:01.025 | INFO     | sidekick.db_config:add_samples:238 - Number of rows inserted: 4114


Created a Database querydb. Inserted sample values from examples/demo/demo_data.csv into table sleep_health_eda, please ask questions!


In [ ]:
# List supported models
list_models()

['h2ogpt-sql-sqlcoder2-4bit',
 'h2ogpt-sql-sqlcoder-34b-alpha-4bit',
 'h2ogpt-sql-nsql-llama-2-7B-4bit',
 'h2ogpt-sql-sqlcoder2',
 'h2ogpt-sql-sqlcoder-7b-2',
 'h2ogpt-sql-sqlcoder-34b-alpha',
 'h2ogpt-sql-nsql-llama-2-7B',
 'gpt-3.5-turbo',
 'gpt-4-8k',
 'gpt-4-1106-preview-128k',
 'gpt-4']

In [ ]:
# Use `list_models()` to check other supported models

def query(
    question: str,
    table_name,
    table_info_path: str,
    sample_qna_path: str,
    regenerate=False,
    regenerate_with_options=False,
):
    """Asks question and returns SQL."""
    base_path = "./"
    # self_correction is enabled by default, set to False if not needed.
    res = ask(
        question=question,
        table_info_path=table_info_path,
        sample_queries_path=sample_qna_path,
        table_name=table_name,
        is_command=False,
        model_name="gpt-4o",  #Other default model option: h2ogpt-sql-sqlcoder-34b-alpha
        is_regenerate=regenerate,
        is_regen_with_options=regenerate_with_options,
        execute_query=False,
        local_base_path=base_path,
    )
    return res

In [ ]:
res = query("What is the average sleep duration for each gender?", table_name="sleep_health_eda",
            table_info_path=table_info_path, sample_qna_path=None)

2024-06-12 16:44:01.044 | INFO     | sidekick.prompter:ask:504 - Table in use: ['sleep_health_eda']
2024-06-12 16:44:01.045 | INFO     | sidekick.prompter:ask:505 - SQL dialect for generation: sqlite
2024-06-12 16:44:01.045 | INFO     | sidekick.prompter:ask:534 - OpenAI key found.
2024-06-12 16:44:01.046 | INFO     | sidekick.prompter:ask:538 - Setting context...
2024-06-12 16:44:01.046 | INFO     | sidekick.prompter:ask:539 - Question: What is the average sleep duration for each gender?
2024-06-12 16:44:01.047 | DEBUG    | sidekick.prompter:ask:557 - Table info path: .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:44:01.048 | INFO     | sidekick.utils:is_resource_low:358 - Number of GPUs: 1
2024-06-12 16:44:01.048 | DEBUG    | sidekick.utils:is_resource_low:362 - Information on device: 0
2024-06-12 16:44:01.744 | INFO     | sidekick.utils:is_resource_low:365 - Total Memory: 7GB
2024-06-12 16:44:01.745 | INFO     | sidekick.utils:is_resource_low:366 - Free GPU memory: 6

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: .//models/sentence_transformers/models--BAAI--bge-base-en/snapshots/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/
Load pretrained SentenceTransformer: .//models/sentence_transformers/models--BAAI--bge-base-en/snapshots/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/



2024-06-12 16:44:02.338 | INFO     | sidekick.utils:_check_file_info:472 - Using information info from path .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:44:02.343 | INFO     | sidekick.query:generate_tasks:354 - Number of context queries found: 0
2024-06-12 16:44:02.343 | INFO     | sidekick.query:generate_tasks:373 - Number of possible contextual queries to question: 0


ValueError: Invalid model name gpt-4o. Available models: dict_keys(['h2ogpt-sql-sqlcoder2-4bit', 'h2ogpt-sql-sqlcoder-34b-alpha-4bit', 'h2ogpt-sql-nsql-llama-2-7B-4bit', 'h2ogpt-sql-sqlcoder2', 'h2ogpt-sql-sqlcoder-7b-2', 'h2ogpt-sql-sqlcoder-34b-alpha', 'h2ogpt-sql-nsql-llama-2-7B', 'gpt-3.5-turbo', 'gpt-4-8k', 'gpt-4-1106-preview-128k', 'gpt-4'])

In [ ]:
print(f"Question = {res[0][0]}")
print("----")
print(f"Generated SQL = {res[0][1]}")

Question = Something went wrong while generating response. Please check the supplied API Keys and try again.
----


IndexError: list index out of range

In [ ]:
# On using re-generation flag we toggle the temperature values between 0 and 1 alternating between low
# (focus/conservative generation and high values (random/creative generation)
res = query("What are the most common occupations among individuals in the dataset?", table_name="sleep_health_eda",
            table_info_path=table_info_path, sample_qna_path=None, regenerate=True)

2024-06-12 16:27:02.898 | INFO     | sidekick.prompter:ask:504 - Table in use: ['sleep_health_eda']
2024-06-12 16:27:02.899 | INFO     | sidekick.prompter:ask:505 - SQL dialect for generation: sqlite
2024-06-12 16:27:02.899 | INFO     | sidekick.prompter:ask:534 - OpenAI key found.
2024-06-12 16:27:02.900 | INFO     | sidekick.prompter:ask:538 - Setting context...
2024-06-12 16:27:02.900 | INFO     | sidekick.prompter:ask:539 - Question: What are the most common occupations among individuals in the dataset?
2024-06-12 16:27:02.901 | DEBUG    | sidekick.prompter:ask:557 - Table info path: .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:27:02.901 | INFO     | sidekick.utils:is_resource_low:358 - Number of GPUs: 1
2024-06-12 16:27:02.902 | DEBUG    | sidekick.utils:is_resource_low:362 - Information on device: 0
2024-06-12 16:27:02.902 | INFO     | sidekick.utils:is_resource_low:365 - Total Memory: 7GB
2024-06-12 16:27:02.903 | INFO     | sidekick.utils:is_resource_low:366 -

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: .//models/sentence_transformers/models--BAAI--bge-base-en/snapshots/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/
Load pretrained SentenceTransformer: .//models/sentence_transformers/models--BAAI--bge-base-en/snapshots/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/



2024-06-12 16:27:03.359 | INFO     | sidekick.utils:_check_file_info:472 - Using information info from path .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:27:03.363 | INFO     | sidekick.query:generate_tasks:353 - Number of context queries found: 0
2024-06-12 16:27:03.364 | INFO     | sidekick.query:generate_tasks:372 - Number of possible contextual queries to question: 0
2024-06-12 16:27:03.365 | INFO     | sidekick.query:_query_tasks:210 - CALLING OPENAI API
2024-06-12 16:27:03.365 | INFO     | sidekick.query:_query_tasks:228 - CALLING OPENAI API


Model Name: ->>>>>>>>>>>>>>> gpt-4
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
1. Scan through the sleep_health_eda table and identify the column name associated with occupations. From the Data section, we see that this column is named "Occupation".
2. Calculate the count of each unique occupation present in the "Occupation" column. This involves grouping the data by occupation and counting the number of entries for each.
3. Sort the calculated counts in descending order. This will allow us to see the occupations with the most counts at the top.
4. Return the occupations and their respective counts from the query. The return type inferred is a list of occupations and their corresponding counts.


2024-06-12 16:27:09.104 | INFO     | sidekick.prompter:ask:607 - Computing user request ...
2024-06-12 16:27:09.105 | DEBUG    | sidekick.query:generate_sql:427 - Query Text:
 
                ### System: Act as a SQL Expert
                # For table ['sleep_health_eda'], given an input *Question*, only generate syntactically correct sqlite SQL queries.
                # Let's work it out in a detailed step by step way using the reasoning from *Tasks* section.
                # Pick the SQL query which has the highest average log probability if more than one result is likely to answer the
                candidate *Question*.
                ### sqlite SQL tables
                ### *Data:* 
For table ['sleep_health_eda'] schema info is mentioned below,

{"Column Name": "Person_ID", "Column Type": "NUMERIC"}
{"Column Name": "Gender", "Column Type": "TEXT", "Sample Values": ["Male", "Female"]}
{"Column Name": "Age", "Column Type": "NUMERIC"}
{"Column Name": "Occupation", "Column Type"

chunk_size_limit is deprecated, please specify chunk_size instead
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:llama_index.indices.struct_store.sql_query:> Table desc str: Schema of table sleep_health_eda:
Table 'sleep_health_eda' has columns: Person_ID (NUMERIC), Gender (TEXT), Age (NUMERIC), Occupation (TEXT), Sleep_Duration (NUMERIC), Quality_of_Sleep (NUMERIC), Physical_Activity_Level (NUMERIC), Stress_Level (NUMERIC), BMI_Category (TEXT), Blood_Pressure (TEXT), Heart_Rate (NUMERIC), Daily_Steps (NUMERIC), Sleep_Disorder (TEXT), and foreign 

2024-06-12 16:27:16.212 | INFO     | sidekick.prompter:ask:612 - Input query: What are the most common occupations among individuals in the dataset?
2024-06-12 16:27:16.213 | INFO     | sidekick.prompter:ask:613 - Generated response:

SELECT "Occupation", COUNT(1) AS "Count" FROM "sleep_health_eda" GROUP BY "Occupation" ORDER BY "Count" DESC LIMIT 100
2024-06-12 16:27:16.214 | INFO     | sidekick.prompter:ask:639 - Alternate responses:

[]


Exiting...


In [ ]:
print(f"Question = {res[0][0]}")
print("----")
print(f"Generated SQL = {res[0][1]}")

Question = I am thinking step by step: 

----
Generated SQL = 1. Scan through the sleep_health_eda table and identify the column name associated with occupations. From the Data section, we see that this column is named "Occupation".
2. Calculate the count of each unique occupation present in the "Occupation" column. This involves grouping the data by occupation and counting the number of entries for each.
3. Sort the calculated counts in descending order. This will allow us to see the occupations with the most counts at the top.
4. Return the occupations and their respective counts from the query. The return type inferred is a list of occupations and their corresponding counts.


In [ ]:
# Alternate options
res = query("What is the average sleep duration for each gender?", table_name="sleep_health_eda",
            table_info_path=table_info_path, sample_qna_path=None, regenerate_with_options=True)

2024-06-12 16:27:16.225 | INFO     | sidekick.prompter:ask:504 - Table in use: ['sleep_health_eda']
2024-06-12 16:27:16.226 | INFO     | sidekick.prompter:ask:505 - SQL dialect for generation: sqlite
2024-06-12 16:27:16.227 | INFO     | sidekick.prompter:ask:534 - OpenAI key found.
2024-06-12 16:27:16.227 | INFO     | sidekick.prompter:ask:538 - Setting context...
2024-06-12 16:27:16.228 | INFO     | sidekick.prompter:ask:539 - Question: What is the average sleep duration for each gender?
2024-06-12 16:27:16.228 | DEBUG    | sidekick.prompter:ask:557 - Table info path: .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:27:16.229 | INFO     | sidekick.utils:is_resource_low:358 - Number of GPUs: 1
2024-06-12 16:27:16.229 | DEBUG    | sidekick.utils:is_resource_low:362 - Information on device: 0
2024-06-12 16:27:16.230 | INFO     | sidekick.utils:is_resource_low:365 - Total Memory: 7GB
2024-06-12 16:27:16.231 | INFO     | sidekick.utils:is_resource_low:366 - Free GPU memory: 5

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: .//models/sentence_transformers/models--BAAI--bge-base-en/snapshots/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/
Load pretrained SentenceTransformer: .//models/sentence_transformers/models--BAAI--bge-base-en/snapshots/b737bf5dcc6ee8bdc530531266b4804a5d77b5d8/



2024-06-12 16:27:16.611 | INFO     | sidekick.utils:_check_file_info:472 - Using information info from path .//var/lib/tmp/sleep_health_eda_table_info.jsonl
2024-06-12 16:27:16.615 | INFO     | sidekick.query:generate_tasks:353 - Number of context queries found: 0
2024-06-12 16:27:16.616 | INFO     | sidekick.query:generate_tasks:372 - Number of possible contextual queries to question: 0
2024-06-12 16:27:16.617 | INFO     | sidekick.query:_query_tasks:210 - CALLING OPENAI API
2024-06-12 16:27:16.617 | INFO     | sidekick.query:_query_tasks:228 - CALLING OPENAI API


Model Name: ->>>>>>>>>>>>>>> gpt-4
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
1. Identify the relevant columns in the question. In this case, we are interested in 'Gender' and 'Sleep_Duration'.
2. Aggregate the data by 'Gender' to split the data into different groups according to gender.
3. For each 'Gender' group, calculate the average 'Sleep_Duration'.
4. Return the 'Gender' along with the corresponding average 'Sleep_Duration' for each gender.


2024-06-12 16:27:20.448 | INFO     | sidekick.prompter:ask:607 - Computing user request ...
2024-06-12 16:27:20.450 | DEBUG    | sidekick.query:generate_sql:427 - Query Text:
 
                ### System: Act as a SQL Expert
                # For table ['sleep_health_eda'], given an input *Question*, only generate syntactically correct sqlite SQL queries.
                # Let's work it out in a detailed step by step way using the reasoning from *Tasks* section.
                # Pick the SQL query which has the highest average log probability if more than one result is likely to answer the
                candidate *Question*.
                ### sqlite SQL tables
                ### *Data:* 
For table ['sleep_health_eda'] schema info is mentioned below,

{"Column Name": "Person_ID", "Column Type": "NUMERIC"}
{"Column Name": "Gender", "Column Type": "TEXT", "Sample Values": ["Male", "Female"]}
{"Column Name": "Age", "Column Type": "NUMERIC"}
{"Column Name": "Occupation", "Column Type"

chunk_size_limit is deprecated, please specify chunk_size instead
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:llama_index.indices.struct_store.sql_query:> Table desc str: Schema of table sleep_health_eda:
Table 'sleep_health_eda' has columns: Person_ID (NUMERIC), Gender (TEXT), Age (NUMERIC), Occupation (TEXT), Sleep_Duration (NUMERIC), Quality_of_Sleep (NUMERIC), Physical_Activity_Level (NUMERIC), Stress_Level (NUMERIC), BMI_Category (TEXT), Blood_Pressure (TEXT), Heart_Rate (NUMERIC), Daily_Steps (NUMERIC), Sleep_Disorder (TEXT), and foreign 

2024-06-12 16:27:23.838 | INFO     | sidekick.prompter:ask:612 - Input query: What is the average sleep duration for each gender?
2024-06-12 16:27:23.838 | INFO     | sidekick.prompter:ask:613 - Generated response:

SELECT "Gender", AVG("Sleep_Duration") AS "Avg_Sleep_Duration" FROM "sleep_health_eda" GROUP BY "Gender" ORDER BY "Avg_Sleep_Duration" DESC
2024-06-12 16:27:23.840 | INFO     | sidekick.prompter:ask:639 - Alternate responses:

[]


Exiting...


In [ ]:
print(f"Question = {res[0][0]}")
print("----Options----")
for _r in res[1]:
    print(_r)

Question = I am thinking step by step: 

----Options----
